# HistGradientBoosting — Model Explainability

SHAP values, MDI / permutation importance, partial dependence, and calibration analysis.

## 0. Configuration

In [ ]:
from datetime import datetime

# ── Path to saved model from random_forest.ipynb ──────────────────────────
MODEL_DIR = "training/saved_models/hist_gradient_boosting"

# ── Data config (must match training) ────────────────────────────────────
DATA_CONFIG = {
    "data_path": r"data/churn_targets.parquet",
    "target": "CHURN_2M",
    "drop_cols": [
        "CARD_ID", "CARD_CANCELLATION_DATE", "CHURN_1M", "CHURN_3M",
        "STATUS", "BIRTH_DT", "OPEN_DT", "MONTH_END", "CUST_ID",
        "LAST_EXPOSURE_DATE", "MONTH_YYYYMM",
    ],
    "random_state": 42,
    "test_month_start": datetime(2025, 2, 1),
    "month_window_start": datetime(2023, 2, 1),
    "month_window_end": datetime(2025, 1, 31),
    "majority_reduction_pct": 0.10,
    "n_lags": 12,
    "priority_non_null_cols": [
        "REMAINING_INSTALLMENT_AMOUNT_lag1",
        "REMAINING_INSTALLMENT_COUNT_lag1",
        "TOTAL_EXPOSURE_AMT_1M_lag1",
        "STMT_PYMNT_AMT_lag1",
        "AVG_TICKET_MTH_lag1",
        "MCC_MOST_FREQUENT_MTH_lag1",
    ],
    "use_streaming_collect": True,
}

# ── MLflow settings (for logging explainability artifacts) ────────────────
EXPERIMENT = "churn-hist-gradient-boosting"
EXPLAINABILITY_RUN_NAME = "hgb-explainability"

# ── SHAP settings ────────────────────────────────────────────────────────
SHAP_SAMPLE_SIZE = 1000
TOP_N_FEATURES = 20
PERMUTATION_REPEATS = 5

## 1. Imports & Setup

In [ ]:
import logging
import os
import sys
import tempfile

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import shap

ROOT = os.path.abspath(".")
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "training"))

from data_processing import load_and_prepare_data
from mlflow_utils import (
    TRACKING_URI,
    configure_training_logging,
    load_model_local,
    start_run,
)

configure_training_logging()
log = logging.getLogger("churn.explain")

## 2. Load Model & Data

In [ ]:
model, metadata = load_model_local(MODEL_DIR)

print(f"Model type   : {metadata['model_type']}")
print(f"Experiment   : {metadata['experiment']}")
print(f"Training run : {metadata['run_id']}")
print(f"Features     : {len(metadata['feature_names'])}")
print(f"Metrics      : {metadata['metrics']}")

In [ ]:
_, X_test_pd, _, y_test, feature_names, data_info = load_and_prepare_data(DATA_CONFIG)

print(f"Test set: {X_test_pd.shape}")

## 3. SHAP Values

In [ ]:
log.info("Computing SHAP values (sample=%s)...", SHAP_SAMPLE_SIZE)

X_sample = X_test_pd.sample(n=min(SHAP_SAMPLE_SIZE, len(X_test_pd)), random_state=42)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

if isinstance(shap_values, list):
    shap_values_pos = shap_values[1]
else:
    shap_values_pos = shap_values

log.info("SHAP values computed. Shape: %s", shap_values_pos.shape)

### 3a. SHAP Summary (Beeswarm) Plot

In [ ]:
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_pos, X_sample, max_display=TOP_N_FEATURES, show=False)
plt.title("SHAP Summary (Beeswarm)")
plt.tight_layout()
plt.show()

### 3b. SHAP Bar Plot (Mean Absolute)

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values_pos, X_sample, plot_type="bar",
    max_display=TOP_N_FEATURES, show=False,
)
plt.title("SHAP Feature Importance (mean |SHAP|)")
plt.tight_layout()
plt.show()

### 3c. SHAP Dependence Plots (Top Features)

In [ ]:
mean_abs_shap = np.abs(shap_values_pos).mean(axis=0)
top_features = pd.Series(mean_abs_shap, index=X_sample.columns).nlargest(5).index.tolist()

for feat in top_features:
    plt.figure(figsize=(8, 5))
    shap.dependence_plot(feat, shap_values_pos, X_sample, show=False)
    plt.title(f"SHAP Dependence: {feat}")
    plt.tight_layout()
    plt.show()

## 4. Built-in Feature Importance

In [ ]:
print(
    "HistGradientBoostingClassifier does not expose `feature_importances_` "
    "(MDI). Use SHAP and permutation importance below."
)

## 5. Permutation Importance

In [ ]:
from sklearn.inspection import permutation_importance

log.info("Computing permutation importance (repeats=%s)...", PERMUTATION_REPEATS)

X_perm_sample = X_test_pd.sample(n=min(2000, len(X_test_pd)), random_state=42)
y_perm_sample = y_test[X_perm_sample.index]

perm_result = permutation_importance(
    model, X_perm_sample, y_perm_sample,
    n_repeats=PERMUTATION_REPEATS, random_state=42, scoring="f1", n_jobs=-1,
)

perm_imp = pd.DataFrame({
    "feature": feature_names,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

print("Top 20 features (Permutation Importance):")
print(perm_imp.head(20).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
plot_perm = perm_imp.head(TOP_N_FEATURES).iloc[::-1]
ax.barh(plot_perm["feature"], plot_perm["importance_mean"], xerr=plot_perm["importance_std"])
ax.set_xlabel("Mean Decrease in F1 (Permutation)")
ax.set_title(f"Top {TOP_N_FEATURES} Features (Permutation Importance)")
fig.tight_layout()
plt.show()

## 6. Partial Dependence Plots

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

numeric_top = [
    f for f in top_features
    if X_test_pd[f].dtype in (np.float32, np.float64, np.int32, np.int64, float, int)
][:4]

if numeric_top:
    X_pdp_sample = X_test_pd.sample(n=min(500, len(X_test_pd)), random_state=42)
    fig, ax = plt.subplots(figsize=(16, 4 * ((len(numeric_top) + 1) // 2)))
    PartialDependenceDisplay.from_estimator(
        model, X_pdp_sample, numeric_top,
        ax=ax, grid_resolution=50,
    )
    fig.suptitle("Partial Dependence Plots", fontsize=14)
    fig.tight_layout()
    plt.show()
else:
    print("No numeric top features for PDP.")

## 7. Calibration Curve

In [ ]:
from sklearn.calibration import CalibrationDisplay

fig, ax = plt.subplots(figsize=(8, 6))
CalibrationDisplay.from_estimator(
    model, X_test_pd, y_test,
    n_bins=20, strategy="uniform",
    ax=ax, name="HistGradientBoosting",
)
ax.set_title("Calibration Curve")
fig.tight_layout()
plt.show()

## 8. Importance Comparison: Permutation vs SHAP

In [ ]:
shap_imp = pd.DataFrame({
    "feature": X_sample.columns,
    "shap": np.abs(shap_values_pos).mean(axis=0),
})

comparison = (
    perm_imp[["feature", "importance_mean"]]
    .rename(columns={"importance_mean": "permutation"})
    .merge(shap_imp, on="feature")
)

for col in ["permutation", "shap"]:
    col_max = comparison[col].max()
    if col_max > 0:
        comparison[f"{col}_norm"] = comparison[col] / col_max

comparison = comparison.sort_values("shap", ascending=False).head(TOP_N_FEATURES)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(comparison))
w = 0.35
ax.bar(x - w / 2, comparison["permutation_norm"], w, label="Permutation (normalised)")
ax.bar(x + w / 2, comparison["shap_norm"], w, label="SHAP (normalised)")
ax.set_xticks(x)
ax.set_xticklabels(comparison["feature"], rotation=45, ha="right", fontsize=7)
ax.set_ylabel("Normalised Importance")
ax.set_title("Feature Importance: Permutation vs SHAP")
ax.legend()
fig.tight_layout()
plt.show()

## 9. Log Explainability Artifacts to MLflow

In [ ]:
with start_run(EXPERIMENT, run_name=EXPLAINABILITY_RUN_NAME) as run:
    mlflow.set_tag("type", "explainability")
    mlflow.set_tag("training_run_id", metadata.get("run_id", ""))

    with tempfile.TemporaryDirectory() as tmp:
        # SHAP summary beeswarm
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values_pos, X_sample, max_display=TOP_N_FEATURES, show=False)
        path = os.path.join(tmp, "shap_summary_beeswarm.png")
        plt.savefig(path, dpi=120, bbox_inches="tight")
        plt.close()
        mlflow.log_artifact(path)

        # SHAP bar
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values_pos, X_sample, plot_type="bar", max_display=TOP_N_FEATURES, show=False)
        path = os.path.join(tmp, "shap_bar.png")
        plt.savefig(path, dpi=120, bbox_inches="tight")
        plt.close()
        mlflow.log_artifact(path)

        # Mean |SHAP| as CSV
        shap_importance = pd.DataFrame({
            "feature": X_sample.columns,
            "mean_abs_shap": np.abs(shap_values_pos).mean(axis=0),
        }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
        csv_path = os.path.join(tmp, "shap_importance.csv")
        shap_importance.to_csv(csv_path, index=False)
        mlflow.log_artifact(csv_path)

        # Permutation importance CSV
        perm_csv = os.path.join(tmp, "permutation_importance.csv")
        perm_imp.to_csv(perm_csv, index=False)
        mlflow.log_artifact(perm_csv)

        # No built-in feature_importances_ for HistGradientBoosting.

        # SHAP dependence for top features
        for feat in top_features:
            plt.figure(figsize=(8, 5))
            shap.dependence_plot(feat, shap_values_pos, X_sample, show=False)
            path = os.path.join(tmp, f"shap_dep_{feat}.png")
            plt.savefig(path, dpi=120, bbox_inches="tight")
            plt.close()
            mlflow.log_artifact(path)

    print(f"Explainability artifacts logged to run: {run.info.run_id}")